# 04 — Hybrid Retrieval

*Notebook 4 of 4. Continues from `03_indexing_and_rag.ipynb` — make sure the processed corpus and Pinecone index from that notebook already exist.*

Dense/vector search is great at "what does this mean," but finance is full of exact tokens — percentages, tickers, guidance ranges — that benefit from plain keyword matching too. This notebook combines both with Reciprocal Rank Fusion and a lightweight reranker. **This is as far as this workshop goes** — tool use and a full agent loop are a separate follow-on session.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data/corpus_manifest.json').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the workshop repo root (looked for data/corpus_manifest.json in this '
        'directory and its parents). Run this notebook from inside Gravitas_Workshop_Starter.'
    )


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workshop root:', ROOT)

from config import load_workshop_env
load_workshop_env()  # load .env once, up front, so later @observe spans don't warn about missing keys

from observability.tracing import flush_langfuse, dashboard_base_url


In [ ]:
chunks_path = ROOT / 'data/processed/chunks.jsonl'
if not chunks_path.exists():
    raise FileNotFoundError(
        'data/processed/chunks.jsonl is missing. Run 03_indexing_and_rag.ipynb first '
        '(Mission 6 builds this file, Mission 7 indexes it into Pinecone) before continuing here.'
    )
print('Processed corpus found:', chunks_path)


## Mission 9 — Hybrid retrieval: finance needs meaning *and* exact strings

### 🧠 Concept

Semantic/vector search is excellent for questions such as **“why did margins weaken?”**

Finance also contains exact tokens that matter disproportionately: **24.3%**, **₹**, quarter names, tickers, guidance ranges and accounting terms.

Hybrid retrieval combines:

- **dense/vector search** for meaning,
- **keyword/BM25 search** for exact lexical matches,
- **fusion** to combine ranked lists,
- **reranking** to spend more compute on a small shortlist.

<img src="../assets/notebook/hybrid_retrieval.png" width="900" alt="Workshop slide showing hybrid dense and keyword retrieval">


### Reciprocal Rank Fusion (RRF)

Dense and keyword systems produce scores on different scales, so comparing raw scores directly can be awkward. RRF instead rewards items that appear near the top of one or more ranked lists.

For each result at rank `r`:

```text
contribution = 1 / (k + r)
```

Then add contributions across rankings.

### ✍️ YOUR TURN 1 — Implement RRF

Write `reciprocal_rank_fusion(...)` right here. Use this mental algorithm:

```text
scores = {}
for each ranking:
    for each source_id with rank starting at 1:
        add 1 / (k + rank)
sort highest score first
```

Then run the sanity check below.


In [ ]:
def reciprocal_rank_fusion(rankings: list[list[str]], *, k: int = 60) -> list[tuple[str, float]]:
    # TODO: score = sum of 1/(k+rank) across rankings; return sorted highest-score-first
    pass

result = reciprocal_rank_fusion([['a', 'b', 'c'], ['b', 'd', 'a']], k=60)
scores = dict(result)
assert result[0][0] in {'a', 'b'}
assert scores['a'] > scores['c'] and scores['b'] > scores['d']
print('reciprocal_rank_fusion: looks correct ✅')
result


### Apply the same idea to real evidence

We will now compare the stages rather than treating “search” as one black box.

> **Teaching note:** `retrieval/ranking.py` uses a deliberately transparent lexical-overlap scorer so you can inspect the ranking stage. Production systems often use a learned reranker.


📎 **Script reference:** `retrieval/search/search.py` (dense + keyword search), `retrieval/search/ranking.py` (`rerank_candidates`, the transparent lexical-overlap reranker), and `retrieval/search/fusion.py` (a working `reciprocal_rank_fusion`, used below instead of the exercise version above so this demo runs regardless) — this is where we assemble the full hybrid pipeline.


In [ ]:
from langfuse import observe
from retrieval.search.search import pinecone_search, keyword_search
from retrieval.search.ranking import rerank_candidates
from retrieval.search.fusion import reciprocal_rank_fusion  # the working version, not the exercise above

query = 'What FY26 revenue growth guidance did Infosys give?'

@observe(name="hybrid-retrieval-checkpoint")
def run_hybrid_retrieval(question):
    dense_hits = pinecone_search(question, top_k=8)
    keyword_hits = keyword_search(question, top_k=8)

    by_id = {h['source_id']: h for h in dense_hits + keyword_hits}
    fused = reciprocal_rank_fusion([
        [h['source_id'] for h in dense_hits],
        [h['source_id'] for h in keyword_hits],
    ])
    fused_hits = [by_id[sid] for sid, _score in fused if sid in by_id]
    final_hits = rerank_candidates(question, fused_hits, top_k=5)
    return dense_hits, keyword_hits, final_hits

dense, keyword, reranked = run_hybrid_retrieval(query)
flush_langfuse()

print('Dense top 3:')
for h in dense[:3]:
    print('-', h['source_id'], h['text'][:120].replace('\n', ' '))

print('\nKeyword top 3:')
for h in keyword[:3]:
    print('-', h['source_id'], h['text'][:120].replace('\n', ' '))

print('\nAfter fusion + reranking:')
for h in reranked:
    print('-', h['source_id'], h['text'][:150].replace('\n', ' '))


### 🔭 TRACE CHECKPOINT 2B — hybrid retrieval

Open `hybrid-retrieval-checkpoint` in Langfuse.

Now the trace should make the retrieval architecture visible:

```text
hybrid-retrieval-checkpoint
├─ pinecone-search
├─ keyword-search
└─ rerank-candidates
```

This is a useful debugging view because **“retrieval failed” is no longer one thing**. Dense search, keyword search, fusion/reranking, or metadata can each be the weak link.

Find one query where the dense and keyword branches disagree. Which branch contributed the evidence you actually wanted?


✅ **Checkpoint:** Look at the dense, keyword, and reranked results printed above. What changed between them — did a chunk that only the keyword search found end up mattering? Explain it out loud to a partner (or OpenCode) before moving on.


---
**That's the end of this workshop's notebook series.** Tool use and a full agent loop (`agent.py` (location TBD by that session), guardrails, skills, a sandbox concept) build on exactly this retrieval stack — that's a separate follow-on session.
